# Image Mosaicking

This notebook creates a mosaic from multiple atmospherically corrected WorldView-3 image tiles.

The workflow is identical for both VNIR and SWIR imagery. The selected sensor is specified in the user settings, and all corrected images available for that sensor are automatically merged into a single georeferenced mosaic while preserving the original spatial reference and metadata.

### Input

- Atmospherically corrected GeoTIFF images located in the corresponding sensor directory (`VNIR` or `SWIR`).

### Output

- A single georeferenced mosaic (`VNIR_mosaic.tif` or `SWIR_mosaic.tif`) saved in the output directory.

In [ ]:
from pathlib import Path
import glob

import rasterio
from rasterio.merge import merge


# Input and output directories
# Update these paths according to your local project structure.

data_dir = Path("../output")


# Mosaic settings

# The code automatically searches for atmospherically corrected images belonging to the selected sensor and merges them into a single mosaic.

sensor = "SWIR"        # "VNIR" or "SWIR"

config = {
    "VNIR": {
        "pattern": "*vnir_corrected*.tif",
        "output": "VNIR_mosaic.tif",
    },
    "SWIR": {
        "pattern": "*swir_corrected*.tif",
        "output": "SWIR_mosaic.tif",
    },
}

cfg = config[sensor]

input_files = sorted(glob.glob(str(data_dir / cfg["pattern"])))

if not input_files:
    raise FileNotFoundError(
        f"No input files matching '{cfg['pattern']}' were found in '{data_dir}'."
    )

print(f"Found {len(input_files)} {sensor} image(s):")
for file in input_files:
    print(Path(file).name)


# Create mosaic

src_files = [rasterio.open(file) for file in input_files]

mosaic, transform = merge(src_files)


# Update metadata

metadata = src_files[0].meta.copy()

metadata.update(
    {
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform,
        "crs": src_files[0].crs,
    }
)



# Save output

output_file = data_dir / cfg["output"]

with rasterio.open(output_file, "w", **metadata) as dst:
    dst.write(mosaic)

print(f"\nMosaic saved to:\n{output_file}")


# Close all opened datasets
for src in src_files:
    src.close()